In [ ]:
# ── Cell 1: Colab GPU Setup ───────────────────────────────────────────────────
# Requires T4 GPU runtime: Runtime → Change runtime type → T4 GPU

import subprocess, sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/fmd'
    IN_COLAB = True
except ImportError:
    PROJECT_DIR = '/Users/aman2394/Desktop/ICWSM-2026-FMD'
    IN_COLAB = False
    print('Running locally — skipping Drive mount')

os.makedirs(f'{PROJECT_DIR}/features', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models/finbert_finetuned', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models/deberta_finetuned', exist_ok=True)
sys.path.insert(0, f'{PROJECT_DIR}/src')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'datasets>=2.19.0', 'accelerate>=0.30.0',
    'scikit-learn>=1.4.0', 'pandas', 'numpy', 'tqdm'
], check=True)

import torch
assert torch.cuda.is_available(), (
    'No GPU detected. Runtime → Change runtime type → T4 GPU'
)
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PROJECT_DIR = {PROJECT_DIR}')

In [ ]:
# ── Cell 2: Load Data (Train Split Only) ─────────────────────────────────────
# Loads train split: original train samples + augmented False samples
# Val and test sets are held out — used only in notebook 06/07.

import sys
sys.path.insert(0, f'{PROJECT_DIR}/src')

import numpy as np
from utils.data_loader import load_combined_data
from utils.data_splitter import get_split_records

orig_records = load_combined_data(PROJECT_DIR)
AUG_PATH = f'{PROJECT_DIR}/data/augmented/augmented_train.json'

train_records, val_records, test_records = get_split_records(
    all_records=orig_records,
    augmented_path=AUG_PATH,
    project_dir=PROJECT_DIR,
)

texts  = [r['text']  for r in train_records]
labels = np.array([int(r['label']) for r in train_records], dtype=np.int32)

n_true  = (labels == 1).sum()
n_false = (labels == 0).sum()
n_aug   = sum(r.get('perturbation_type') is not None for r in train_records)

print(f'Train set  : {len(texts)} samples')
print(f'  True     : {n_true}')
print(f'  False    : {n_false}  (of which {n_aug} are augmented)')
print(f'  Balance  : {labels.mean():.2%} true')
print(f'Val set    : {len(val_records)} samples (held out)')
print(f'Test set   : {len(test_records)} samples (held out)')
print(f'\nSample text: {texts[0][:200]}')


In [ ]:
# ── Cell 3: OOF Fine-Tuning with FinBERT ─────────────────────────────────────
# Model : ProsusAI/finbert (110M params — fits comfortably on T4)
# Output: oof_preds_finbert — shape (N, 2) softmax probs [P(False), P(True)]
# Est.  : ~30 min on T4 (5 folds × ~6 min each)

from features.tier2_encoder import run_oof_finetuning

FINBERT_CONFIG = dict(
    model_name='ProsusAI/finbert',
    output_dir=f'{PROJECT_DIR}/models/finbert_finetuned',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    eval_strategy='epoch',
    save_strategy='epoch',
    report_to='none',
    n_splits=5,
    random_state=42,
)

print('Running 5-fold OOF fine-tuning with FinBERT ...')
oof_preds_finbert = run_oof_finetuning(
    texts=texts,
    labels=labels,
    config=FINBERT_CONFIG,
)

print(f'FinBERT OOF preds shape : {oof_preds_finbert.shape}')  # (N, 2)

import numpy as np
np.save(f'{PROJECT_DIR}/feature_cache/_oof_finbert.npy', oof_preds_finbert)
print('FinBERT OOF predictions saved.')

# Quick accuracy check
from sklearn.metrics import accuracy_score
oof_preds_finbert_class = oof_preds_finbert.argmax(axis=1)
acc = accuracy_score(labels, oof_preds_finbert_class)
print(f'FinBERT OOF accuracy : {acc:.4f}')

In [ ]:
# ── Cell 4: OOF Fine-Tuning with DeBERTa-v3-large ────────────────────────────
# Model : microsoft/deberta-v3-large (304M params — needs memory optimizations)
# Output: oof_preds_deberta — shape (N, 2) softmax probs
# Est.  : ~2–3 hrs on T4
#
# OOM FALLBACK: If DeBERTa-v3-large causes CUDA OOM:
#   1. Reduce per_device_train_batch_size to 2
#   2. Increase gradient_accumulation_steps to 8
#   3. If still OOM → switch MODEL_NAME to 'microsoft/deberta-v3-base' (86M params)

from features.tier2_encoder import run_oof_finetuning

DEBERTA_MODEL_NAME = 'microsoft/deberta-v3-large'
# DEBERTA_MODEL_NAME = 'microsoft/deberta-v3-base'  # ← OOM fallback

DEBERTA_CONFIG = dict(
    model_name=DEBERTA_MODEL_NAME,
    output_dir=f'{PROJECT_DIR}/models/deberta_finetuned',
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # Effective batch = 16
    learning_rate=1e-5,
    fp16=True,
    gradient_checkpointing=True,     # Required for 304M params on T4
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    eval_strategy='epoch',
    save_strategy='epoch',
    report_to='none',
    n_splits=5,
    random_state=42,
)

print(f'Running 5-fold OOF fine-tuning with {DEBERTA_MODEL_NAME} ...')
oof_preds_deberta = run_oof_finetuning(
    texts=texts,
    labels=labels,
    config=DEBERTA_CONFIG,
)

print(f'DeBERTa OOF preds shape : {oof_preds_deberta.shape}')  # (N, 2)

import numpy as np
np.save(f'{PROJECT_DIR}/feature_cache/_oof_deberta.npy', oof_preds_deberta)
print('DeBERTa OOF predictions saved.')

from sklearn.metrics import accuracy_score
acc = accuracy_score(labels, oof_preds_deberta.argmax(axis=1))
print(f'DeBERTa OOF accuracy : {acc:.4f}')

In [ ]:
# ── Cell 5: Concatenate OOF Predictions → Save tier2_oof_preds.npy ───────────

import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Load from checkpoints if cells were run in separate sessions
oof_preds_finbert = np.load(f'{PROJECT_DIR}/feature_cache/_oof_finbert.npy')
oof_preds_deberta = np.load(f'{PROJECT_DIR}/feature_cache/_oof_deberta.npy')

print('OOF shapes before concat:')
print(f'  FinBERT  : {oof_preds_finbert.shape}')  # (N, 2)
print(f'  DeBERTa  : {oof_preds_deberta.shape}')  # (N, 2)

# Concatenate: [P_fin(False), P_fin(True), P_deb(False), P_deb(True)]
tier2_oof_preds = np.hstack([oof_preds_finbert, oof_preds_deberta])
print(f'\nTier 2 combined shape : {tier2_oof_preds.shape}')  # (N, 4)

# Final accuracy reports
labels_loaded = np.load(f'{PROJECT_DIR}/feature_cache/labels.npy')

for name, preds in [('FinBERT', oof_preds_finbert), ('DeBERTa', oof_preds_deberta)]:
    y_pred = preds.argmax(axis=1)
    acc = accuracy_score(labels_loaded, y_pred)
    f1  = f1_score(labels_loaded, y_pred, average='binary')
    print(f'{name:10s}  accuracy={acc:.4f}  f1={f1:.4f}')

# Save
np.save(f'{PROJECT_DIR}/feature_cache/tier2_oof_preds.npy', tier2_oof_preds)

print('\n✅ Saved:')
print(f'  {PROJECT_DIR}/feature_cache/tier2_oof_preds.npy — shape {tier2_oof_preds.shape}')

assert not np.isnan(tier2_oof_preds).any(), 'NaN values detected in tier2_oof_preds!'
print('NaN check passed.')